In [1]:
import re
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from collections import Counter

In [37]:
# Ensure necessary resources are downloaded
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Admin\AppData\Roaming\nltk_data...
[nltk_data]   Unzipping tokenizers\punkt_tab.zip.


True

In [38]:
data = pd.read_csv("Data.csv")

In [39]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7339 entries, 0 to 7338
Data columns (total 1 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Data    7339 non-null   object
dtypes: object(1)
memory usage: 57.5+ KB


In [40]:
data.head()

,Data
0,Watch or listen live weekdays at 8:30am MT at ...
1,Watch or listen live weekdays at 8:30am MT at ...
2,"Chubby And Hot, Always Stir The Pot!"
3,"Chubby And Hot, Always Stir The Pot!"
4,"Journalist, publisher of Rebel News — telling ..."


In [41]:
# Rename the column for simplicity
data.rename(columns={"Data": "record"}, inplace=True)

In [42]:
# Define patterns
pattern_date_no_alphabets = r'\b\d{1,4}[-/]\d{1,2}[-/]\d{1,4}\b'
pattern_word_start_w = r'\b[wW]\w*\b'
pattern_word_not_url = r'\b[a-zA-Z](?!://)\w*\b'
pattern_emojis = r':\)|:D|;\)|:P'
pattern_decimal = r'\b\d+\.\d+\b'
pattern_ip = r'\b(?:\d{1,3}\.){3}\d{1,3}\b'
pattern_new_line = r'\n'
pattern_hashtag = r'#\w+'
pattern_non_alphanumeric = r'[^a-zA-Z0-9\s]'
pattern_url = r'https?://\S+'

## Question 1: Count records with dates expressed without alphabets

In [43]:
data['date_no_alphabets'] = data['record'].apply(lambda x: bool(re.search(pattern_date_no_alphabets, str(x))))
count_date_no_alphabets = data['date_no_alphabets'].sum()
print("1. Records with dates expressed without alphabets:", count_date_no_alphabets)

1. Records with dates expressed without alphabets: 4


## Question 2: Count records with words starting with "w"

In [44]:
data['word_start_w'] = data['record'].apply(lambda x: len(re.findall(pattern_word_start_w, str(x))))
count_word_start_w = data['word_start_w'].sum()
print("2. Records with words starting with 'w':", count_word_start_w)

2. Records with words starting with 'w': 5584


## Question 3: Count records with words starting with an alphabet but not a URL

In [45]:
data['word_not_url'] = data['record'].apply(lambda x: len(re.findall(pattern_word_not_url, str(x))))
count_word_not_url = data['word_not_url'].sum()
print("3. Records with words starting with an alphabet but not a URL:", count_word_not_url)

3. Records with words starting with an alphabet but not a URL: 119395


## Question 4: Count tweets containing specific emojis

In [46]:
data['emojis'] = data['record'].apply(lambda x: len(re.findall(pattern_emojis, str(x))))
count_emojis = data['emojis'].sum()
print("4. Tweets containing specific emojis:", count_emojis)

4. Tweets containing specific emojis: 28


## Question 5: Count records with decimal numbers

In [47]:
data['decimal_numbers'] = data['record'].apply(lambda x: len(re.findall(pattern_decimal, str(x))))
count_decimal_numbers = data['decimal_numbers'].sum()
print("5. Records with decimal numbers:", count_decimal_numbers)

5. Records with decimal numbers: 36


## Question 6: Count total IP addresses across all records

In [48]:
data['ip_addresses'] = data['record'].apply(lambda x: len(re.findall(pattern_ip, str(x))))
total_ip_addresses = data['ip_addresses'].sum()
print("6. Total IP addresses:", total_ip_addresses)

6. Total IP addresses: 0


## Question 7: Count records with a new line

In [49]:
data['new_line'] = data['record'].apply(lambda x: bool(re.search(pattern_new_line, str(x))))
count_new_line = data['new_line'].sum()
print("7. Records with a new line:", count_new_line)

7. Records with a new line: 1211


## Question 8: Count total hashtags across all tweets

In [50]:
data['hashtags'] = data['record'].apply(lambda x: len(re.findall(pattern_hashtag, str(x))))
total_hashtags = data['hashtags'].sum()
print("8. Total hashtags:", total_hashtags)

8. Total hashtags: 2924


## Question 9: Substitute all non-alphanumeric characters with a new line

In [51]:
data['substituted_text'] = data['record'].apply(lambda x: re.sub(pattern_non_alphanumeric, '\n', str(x)))
print("9. Substituted text for all non-alphanumeric characters created.")

9. Substituted text for all non-alphanumeric characters created.


## Question 10: Count total URLs across all tweets

In [52]:
data['urls'] = data['record'].apply(lambda x: len(re.findall(pattern_url, str(x))))
total_urls = data['urls'].sum()
print("10. Total URLs:", total_urls)

10. Total URLs: 4


## Perform stemming and lemmatization

In [55]:
stop_words = set(stopwords.words('english'))
porter = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [56]:
# Tokenize records
data['tokens'] = data['record'].apply(lambda x: word_tokenize(str(x)))

In [57]:
# Remove stopwords
data['tokens_no_stopwords'] = data['tokens'].apply(lambda tokens: [t for t in tokens if t.lower() not in stop_words])

In [58]:
# Apply stemming
data['stemmed_tokens'] = data['tokens_no_stopwords'].apply(lambda tokens: [porter.stem(t) for t in tokens])
unique_stemmed_tokens = set(sum(data['stemmed_tokens'], []))

In [59]:
# Apply lemmatization
data['lemmatized_tokens'] = data['tokens_no_stopwords'].apply(lambda tokens: [lemmatizer.lemmatize(t) for t in tokens])
unique_lemmatized_tokens = set(sum(data['lemmatized_tokens'], []))

In [62]:
# Word frequency analysis after stemming and lemmatization
stemmed_freq = Counter(sum(data['stemmed_tokens'], []))
lemmatized_freq = Counter(sum(data['lemmatized_tokens'], []))

## Top 10 words after stemming and lemmatization

In [63]:
top_10_stemmed = stemmed_freq.most_common(10)
top_10_lemmatized = lemmatized_freq.most_common(10)

## Display stemming and lemmatization results

In [64]:
print("\nStemming and Lemmatization Results:")
print("Unique tokens before stemming:", len(unique_stemmed_tokens))
print("Unique tokens after stemming:", len(unique_stemmed_tokens))
print("Unique tokens before lemmatization:", len(unique_lemmatized_tokens))
print("Unique tokens after lemmatization:", len(unique_lemmatized_tokens))
print("Top 10 words after stemming:", top_10_stemmed)
print("Top 10 words after lemmatization:", top_10_lemmatized)


Stemming and Lemmatization Results:
Unique tokens before stemming: 7437
Unique tokens after stemming: 7437
Unique tokens before lemmatization: 9977
Unique tokens after lemmatization: 9977
Top 10 words after stemming: [('.', 10266), (',', 9314), ('#', 2936), ('&', 1699), ('@', 1620), ('|', 1231), ('alberta', 1061), ('!', 972), ('outdoor', 682), ('’', 660)]
Top 10 words after lemmatization: [('.', 10266), (',', 9314), ('#', 2936), ('&', 1699), ('@', 1620), ('|', 1231), ('Alberta', 1051), ('!', 972), ('’', 660), (':', 625)]
